In [1]:
import pandas as pd
from collections import defaultdict
import time

# ------------------------------
# Чтение и парсинг данных
# ------------------------------
def parse_row(row):
    """Парсинг строки CSV"""
    id1, id2 = map(int, row['ids'].split('_'))
    size1, size2, intersect = map(int, row['sizes'].split('_'))
    return id1, id2, size1, size2, intersect == 1

def load_conditions(file_path):
    """Загружаем данные и создаем список условий и всех id"""
    df = pd.read_csv(file_path)
    conditions = []
    all_set_ids = set()
    for _, row in df.iterrows():
        id1, id2, size1, size2, should_intersect = parse_row(row)
        conditions.append((id1, id2, size1, size2, should_intersect))
        all_set_ids.add(id1)
        all_set_ids.add(id2)
    all_set_ids = sorted(all_set_ids)
    return conditions, all_set_ids

# ------------------------------
# Поиск компонент через DFS
# ------------------------------
def dfs(u, graph, comp, cid):
    comp[u] = cid
    for v in graph[u]:
        if comp[v] == -1:
            dfs(v, graph, comp, cid)

# ------------------------------
# Основной алгоритм пересечений
# ------------------------------
def solve_component_intersection(conditions, all_set_ids):
    n = len(all_set_ids)
    id_to_idx = {id_: i for i, id_ in enumerate(all_set_ids)}
    idx_to_id = {i: id_ for i, id_ in enumerate(all_set_ids)}

    # Определяем целевые размеры множеств
    target_size = [0]*n
    for id1, id2, size1, size2, _ in conditions:
        i, j = id_to_idx[id1], id_to_idx[id2]
        target_size[i] = max(target_size[i], size1)
        target_size[j] = max(target_size[j], size2)

    # Строим граф пересечений
    graph = [[] for _ in range(n)]
    for id1, id2, _, _, should_intersect in conditions:
        if should_intersect:
            i, j = id_to_idx[id1], id_to_idx[id2]
            graph[i].append(j)
            graph[j].append(i)

    # Находим компоненты
    comp = [-1]*n
    cid = 0
    for i in range(n):
        if comp[i] == -1:
            dfs(i, graph, comp, cid)
            cid += 1

    # Создаем множества
    sets = [set() for _ in range(n)]
    next_elem = 1

    for c in range(cid):
        nodes = [i for i in range(n) if comp[i] == c]
        max_size = max(target_size[i] for i in nodes)
        # общий пул элементов
        pool = list(range(next_elem, next_elem + max_size))
        next_elem += max_size

        # раздаём элементы
        for u in nodes:
            take = min(len(pool), target_size[u])
            sets[u].update(pool[:take])
            while len(sets[u]) < target_size[u]:
                sets[u].add(next_elem)
                next_elem += 1

    # Конвертируем обратно в словарь
    result = {idx_to_id[i]: sets[i] for i in range(n)}
    return result

# ------------------------------
# Вычисление точности (accuracy)
# ------------------------------
def evaluate_solution(sets_dict, conditions):
    correct = 0
    for id1, id2, size1, size2, should_intersect in conditions:
        set1, set2 = sets_dict[id1], sets_dict[id2]
        size_correct = (len(set1) == size1) and (len(set2) == size2)
        intersect_correct = (len(set1 & set2) > 0) == should_intersect
        if size_correct and intersect_correct:
            correct += 1
    return correct / len(conditions)

# ------------------------------
# Статистика уникальных элементов
# ------------------------------
def calculate_statistics(sets_dict):
    all_elements = set()
    total_elements = 0
    for s in sets_dict.values():
        all_elements.update(s)
        total_elements += len(s)
    return len(all_elements), total_elements

# ------------------------------
# Пример использования
# ------------------------------
start_total = time.time()

# Путь к файлу Kaggle
file_path = '/kaggle/input/triplets-all/triplets_all.csv'
conditions, all_set_ids = load_conditions(file_path)

print(f"Всего множеств: {len(all_set_ids)}")
print(f"Всего условий: {len(conditions)}")

start = time.time()
solution = solve_component_intersection(conditions, all_set_ids)
end = time.time()
print(f"Время выполнения алгоритма: {end - start:.3f} сек")

accuracy = evaluate_solution(solution, conditions)
unique_elements, total_elements = calculate_statistics(solution)

print(f"Accuracy (точность): {accuracy*100:.4f}%")
print(f"Уникальных элементов: {unique_elements}")
print(f"Сумма размеров множеств: {total_elements}")

print(f"Общее время: {time.time() - start_total:.3f} сек")



Всего множеств: 878
Всего условий: 385003
Время выполнения алгоритма: 0.252 сек
Accuracy (точность): 91.9785%
Уникальных элементов: 1118
Сумма размеров множеств: 78925
Общее время: 19.091 сек


In [2]:
def save_solution_exact_format(solution, conditions, path):
    """
    solution: dict[int, set[int]]
        {set_id: set(elements)}
    conditions: list of tuples
        (id1, id2, size1, size2, should_intersect)
    """

    data = []

    for id1, id2, _, _, _ in conditions:
        s1 = solution[id1]
        s2 = solution[id2]

        size1 = len(s1)
        size2 = len(s2)
        intersect = 1 if (s1 & s2) else 0

        data.append([
            f"{id1}_{id2}",
            f"{size1}_{size2}_{intersect}"
        ])

    df = pd.DataFrame(data, columns=["ids", "sizes"])
    df.to_csv(path, index=False)

In [3]:
save_solution_exact_format(
    solution,
    conditions,
    "/kaggle/working/solution.csv"
)